In [13]:
%load_ext autoreload
%autoreload 2

try:
    from pdfminer.high_level import extract_text
except:
    !pip install pdfminer.six

In [7]:
resume_text = extract_text("test_resume_cn.pdf")

print(resume_text)

电子邮箱: yicong.xiao@u.nus.edu

手机: +86 18026919165

GitHub: https://github.com/yxiaoaz 领英: https://www.linkedin.com/in/edwardxiao2001/

肖亦聪

教育经历
新加坡国立大学
计算机科学（人工智能方向） 硕士
 GPA: 4.9 / 5.0
 院长嘉许名单成员（专业前 5%-10%）
 相关课程：不确定性建模、神经网络与深度学习、多媒体计算理论、图机器学习、大数据技术

2024 年 8 月 - 2026 年 2 月

香港科技大学
计量金融与计算机科学 本科
 GPA: 3.6 / 4.3
 院长嘉许名单成员
 相关课程：机器学习、概率论、Python 编程与数据科学、时间序列分析、搜索引擎算法、面向对象程序设计

2019 年 9 月-2024 年 1 月

工作经历
Intact Financial (HK) Limited
数据科学实习生
 与团队合作开发了一款针对保险文件的智能文档解析软件，从 word 或 PDF 格式的保险文件中提取非结构化数据，并根

2024 年 1 月- 2024 年 5 月

据业务逻辑进行数据清洗和加工，输出结构化的保单数据，减少了核保人员手动记录新保单的工作量。

 设计并实现了一个针对保险文件的聚类与主题建模算法，能够根据文本布局和语义内容对大批量保险文件进行聚类，并
通过每个类中所有文本的 TF-IDF 值、在文件中出现的位置等因素对于该类进行主题建模，提高了针对来自新市场的大
批量未知文档的文本解析逻辑开发的效率。

 对来自新市场的法语保险文件进行探索性数据分析，与核保部门持续对接沟通开发需求，使用 Python 开发了公司首个

针对法语文件的文档解析和数据加工程序。

 使用 Python 和 Jinja 开发了一个能够自动生成仿真保险文件数据的程序，实现了文档解析 AI 模型的高效测试与迭代。

中国电信（北京研究院）
算法实习生
 研究针对法律文本的知识表征学习，对于摘要生成、实体链接等领域的自然语言模型进行复现、实验以及分析。
 使用 PyTorch 和 Hugging Face 对大语言模型进行微调，在包括提示词工程（prompt engineering)

In [10]:
import json
from openai import OpenAI
from datetime import datetime

DEEPSEEK_API_KEY = 'sk-135fe459060d4443ab30b8ae1f68f900'

client = OpenAI(
    api_key= DEEPSEEK_API_KEY ,
    base_url="https://api.deepseek.com",
)

system_prompt = """
```xml
<instruction>
你是一个求职助手，需要从用户的简历文本中提取出关键信息，并以结构化的 JSON 格式返回结果。请按照以下步骤处理输入内容：

1. 阅读并理解用户提供的简历文本内容。
2. 从中提取出以下信息（如果信息不存在，对应字段的内容请设为 None）：
  - 教育背景 (education)，格式为列表，每项包含学校 (school)、学位 (degree)、专业 (major)、毕业年份(graduation_year)
  - 工作经历 (work_experience)，格式为列表，每项包含公司名称 (company)、职位 (position)、职责描述 (responsibilities)
    - company、position 字段皆为字符串，请完整提取相关信息
    - responsibilities 字段为列表，列表包含对于当前经历的工作内容的关键信息抽取
    - work_experience 既包含正式工作记录，也包含实习、兼职等其它形式的工作经历，请尽量全面涵盖
  - 技能 (skills)，格式为字符串列表
    - 请先检查简历中是否包含专门的 “技能” 或类似的内容块，从这些内容中抽取关键词并添加进列表中
    - 用户的工作或项目经历中提到的技能，也需要添加进当前列表中

3. 将提取的信息按照指定的JSON结构组织并返回，不包含任何额外文本或说明。
4. 返回结果只能包含以上提及的字段，不能包含任何其他字段，请严格遵循说明
</instruction>

<example>
<input>
张三  
电话：13800001111  
邮箱：zhangsan@example.com  

教育背景：  
南京大学，计算机科学，学士，2015.09 - 2019.06  

工作经历：  
阿里巴巴，后端开发工程师，2019.07 - 至今  
- 负责电商平台后端服务开发  
- 参与高并发系统架构设计  

实习经历：  
Intact Financial Corp
数据科学家 2020.03 - 2020.12  
- 利用 PyTorch 实现了基于用户行为的推荐算法
- 提升点击率15%  

技能：Java, Python, MySQL, Redis
</input>

<output>
{
  "education": [
    {
      "school": "南京大学",
      "degree": "学士",
      "major": "计算机科学",
      "graduation_year": 2019,
    }
  ],
  "work_experience": [
    {
      "company": "阿里巴巴",
      "position": "后端开发工程师",
      "responsibilities': ["电商平台后端服务开发", "高并发系统架构设计"]
    },
    {
      "company": "Intact Financial Corp",
      "position": "数据科学家",
      "responsibilities': ["推荐算法研发"]
    }
  ],
  "skills": ["Java", "Python", "MySQL", "Redis", "PyTorch", "后端开发", "架构设计"]
  ""
</output>
</example>

</instruction>
```
"""


messages = [{"role": "system", "content": system_prompt},
            {"role": "user", "content": resume_text}]

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
    response_format={
        'type': 'json_object'
    }
)

parse_result = json.loads(response.choices[0].message.content)

In [11]:
parse_result

{'education': [{'school': '新加坡国立大学',
   'degree': '硕士',
   'major': '计算机科学（人工智能方向）',
   'graduation_year': 2026},
  {'school': '香港科技大学',
   'degree': '本科',
   'major': '计量金融与计算机科学',
   'graduation_year': 2024}],
 'work_experience': [{'company': 'Intact Financial (HK) Limited',
   'position': '数据科学实习生',
   'responsibilities': ['开发智能文档解析软件，提取非结构化数据并进行数据清洗和加工',
    '设计并实现保险文件的聚类与主题建模算法',
    '对法语保险文件进行探索性数据分析并开发文档解析程序',
    '开发自动生成仿真保险文件数据的程序']},
  {'company': '中国电信（北京研究院）',
   'position': '算法实习生',
   'responsibilities': ['研究法律文本的知识表征学习，复现、实验以及分析自然语言模型',
    '使用PyTorch和Hugging Face对大语言模型进行微调',
    '开发用于获取训练数据的Python程序',
    '管理并协调多个数据标注项目']}],
 'skills': ['Python',
  'SQL',
  'Java',
  'C++',
  'PyTorch',
  'NumPy',
  'Pandas',
  'Scikit-learn',
  'OpenCV',
  'PowerBI',
  'Matplotlib',
  'Seaborn',
  'AWS S3',
  'Spark',
  'Databricks',
  'MongoDB',
  'Git',
  'GitHub',
  'Data Version Control (DVC)']}

In [14]:
from app.services.llm.open_ai_service_provider import openAIServiceProvider

ds_llm = openAIServiceProvider(api_url = "https://api.deepseek.com", api_key = DEEPSEEK_API_KEY)

In [15]:
messages = [{"role": "system", "content": system_prompt},
            {"role": "user", "content": resume_text}]
other_prompt_args={
    "response_format":{
        'type': 'json_object'
    }
}
ds_llm.get_completion(model_name="deepseek-chat", messages=messages, other_prompt_args=other_prompt_args )

{'education': [{'school': '新加坡国立大学',
   'degree': '硕士',
   'major': '计算机科学（人工智能方向）',
   'graduation_year': 2026},
  {'school': '香港科技大学',
   'degree': '本科',
   'major': '计量金融与计算机科学',
   'graduation_year': 2024}],
 'work_experience': [{'company': 'Intact Financial (HK) Limited',
   'position': '数据科学实习生',
   'responsibilities': ['开发智能文档解析软件，提取非结构化数据并进行数据清洗和加工',
    '设计并实现保险文件的聚类与主题建模算法',
    '对法语保险文件进行探索性数据分析并开发文档解析和数据加工程序',
    '开发自动生成仿真保险文件数据的程序']},
  {'company': '中国电信（北京研究院）',
   'position': '算法实习生',
   'responsibilities': ['研究法律文本的知识表征学习',
    '使用PyTorch和Hugging Face对大语言模型进行微调',
    '开发用于获取训练数据的Python程序',
    '管理并协调多个数据标注项目']}],
 'skills': ['Python',
  'SQL',
  'Java',
  'C++',
  'PyTorch',
  'NumPy',
  'Pandas',
  'Scikit-learn',
  'OpenCV',
  'PowerBI',
  'Matplotlib',
  'Seaborn',
  'AWS S3',
  'Spark',
  'Databricks',
  'MongoDB',
  'Git',
  'GitHub',
  'Data Version Control (DVC)']}

In [ ]:
import base64

# binary to base64
with open("test_resume_cn.pdf", 'rb') as file:
    blob = base64.b64encode(file.read())

print(blob)

# base64 back to binary and then to file
with open('encode-decode.pdf', 'wb') as file:
    file.write(base64.b64decode(blob))



b'...'


In [ ]:
from app.models.base import Base
from sqlalchemy import Column, JSON, Text, Integer, ForeignKey

class Resume(Base):
    __tablename__ = 'resume_db'

    id = Column(Integer, primary_key=True)

    user_id = Column(Integer, ForeignKey("user.id"))
    source_file_blob = Column(Text)
    extracted_content = Column(JSON)